# Diagnóstico mínimo de 3D Gaussian Splatting

Este notebook no ejecuta el experimento completo. Comprueba, en orden, el entorno CUDA/PyTorch, las extensiones compiladas, el formato COLMAP y un entrenamiento de 10 iteraciones.

Si falla, la última celda clasifica el error y deja el log completo en `/content/gs_diagnostic_train.log`. Ejecutarlo en una sesión nueva de Colab con GPU habilitada.

In [1]:
from pathlib import Path
import os, sys, subprocess, tempfile, json, struct, platform, shutil

# Ajusta solo esta ruta si la detección automática elige otra escena.
GS_REPO = Path('/content/gaussian-splatting')
SCENE_FOLDER = None  # Ejemplo: Path('/content/original')
DIAGNOSTIC_ITERATIONS = 10
LOG_PATH = Path('/content/gs_diagnostic_train.log')
print('Python:', sys.executable)
print('Repositorio esperado:', GS_REPO)

Python: /usr/bin/python3
Repositorio esperado: /content/gaussian-splatting


In [2]:
print('=== Entorno ===')
print('Versión:', sys.version.replace('\n', ' '))
print('Sistema:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA de PyTorch:', torch.version.cuda)
    print('CUDA disponible:', torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print('GPU:', props.name)
        print('Compute capability:', f'{props.major}.{props.minor}')
        print('VRAM GB:', round(props.total_memory / 1024**3, 2))
        x = torch.randn((8, 8), device='cuda')
        print('Tensor CUDA: OK', x.mean().item())
    else:
        print('DIAGNÓSTICO: Colab no tiene GPU activa. Activa Entorno de ejecución > Cambiar tipo de entorno > T4 GPU.')
except Exception as error:
    print('ERROR PyTorch/CUDA:', repr(error))

=== Entorno ===
Versión: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
Sistema: Linux-6.6.122+-x86_64-with-glibc2.35
PyTorch: 2.6.0+cu124
CUDA de PyTorch: 12.4
CUDA disponible: True
GPU: Tesla T4
Compute capability: 7.5
VRAM GB: 14.56
Tensor CUDA: OK 0.16585317254066467


In [3]:
print('=== Repositorio y extensiones ===')
if not GS_REPO.exists():
    raise FileNotFoundError('No existe /content/gaussian-splatting. Ejecuta primero la celda de preparación del notebook principal.')
print('Repositorio:', GS_REPO)
print(subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=GS_REPO, text=True, capture_output=True).stdout.strip())
for module_name in ['diff_gaussian_rasterization', 'simple_knn._C']:
    try:
        __import__(module_name)
        print(module_name, ': OK')
    except Exception as error:
        print(module_name, ': ERROR ->', repr(error))
        print('Esto indica una incompatibilidad entre PyTorch, CUDA y la compilación de la extensión.')
try:
    from diff_gaussian_rasterization import GaussianRasterizationSettings, GaussianRasterizer
    from simple_knn._C import distCUDA2
    print('Importaciones usadas por train.py: OK')
except Exception as error:
    raise RuntimeError('Las extensiones no están utilizables; no tiene sentido lanzar train.py todavía.') from error

=== Repositorio y extensiones ===
Repositorio: /content/gaussian-splatting
54c035f
diff_gaussian_rasterization : OK
simple_knn._C : OK
Importaciones usadas por train.py: OK


In [4]:
print('=== Dataset COLMAP ===')
if SCENE_FOLDER is None:
    candidates = []
    for images_bin in Path('/content').glob('**/sparse/0/images.bin'):
        candidate = images_bin.parents[2]
        if (candidate / 'images').is_dir():
            candidates.append(candidate)
    if not candidates:
        raise FileNotFoundError('No encontré una escena con images/ y sparse/0/images.bin bajo /content.')
    print('Escenas detectadas:')
    for candidate in candidates:
        print(' -', candidate)
    SCENE_FOLDER = candidates[0]
SCENE_FOLDER = Path(SCENE_FOLDER)
images_dir = SCENE_FOLDER / 'images'
images_bin = SCENE_FOLDER / 'sparse' / '0' / 'images.bin'
image_files = sorted([p for p in images_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}])
print('Escena:', SCENE_FOLDER)
print('Imágenes:', len(image_files))
print('images.bin:', images_bin.exists())
if not image_files or not images_bin.exists():
    raise RuntimeError('La escena no tiene el formato COLMAP esperado.')
with open(images_bin, 'rb') as file:
    n_poses = struct.unpack('<Q', file.read(8))[0]
print('Poses COLMAP:', n_poses)
print('Primera imagen:', image_files[0].name, image_files[0].stat().st_size, 'bytes')
if n_poses == 0:
    raise RuntimeError('images.bin no contiene poses.')
if len(image_files) < 2:
    raise RuntimeError('Se necesitan al menos dos vistas para esta prueba.')
print('Formato básico: OK')

=== Dataset COLMAP ===
Escenas detectadas:
 - /content/truck
 - /content/drive/MyDrive/inf479/proyecto_final_resultados/truck_hadamard_025
 - /content/drive/MyDrive/inf479/proyecto_final_resultados/images_baseline
 - /content/drive/MyDrive/inf479/proyecto_final_resultados/images_spc_005
 - /content/drive/MyDrive/inf479/proyecto_final_resultados/images_spc_010
 - /content/drive/MyDrive/inf479/proyecto_final_resultados/images_spc_025
 - /content/drive/MyDrive/inf479/proyecto_final_resultados/images_spc_050
 - /content/drive/MyDrive/inf479/proyecto_final_resultados/pilot_iter1500/images_original
 - /content/drive/MyDrive/inf479/proyecto_final_resultados/pilot_iter1500/images_spc_025
 - /content/drive/MyDrive/inf479/tandt_db_extracted/db/drjohnson
 - /content/drive/MyDrive/inf479/tandt_db_extracted/db/playroom
 - /content/drive/MyDrive/inf479/tandt_db_extracted/tandt/train
 - /content/drive/MyDrive/inf479/tandt_db_extracted/tandt/truck
Escena: /content/truck
Imágenes: 251
images.bin: True


In [5]:
print('=== Entrenamiento mínimo ===')
model_path = Path(tempfile.mkdtemp(prefix='gs_diagnostic_', dir='/content'))
command = [
    sys.executable, 'train.py',
    '-s', str(SCENE_FOLDER), '-m', str(model_path),
    '--iterations', str(DIAGNOSTIC_ITERATIONS),
    '--test_iterations', str(DIAGNOSTIC_ITERATIONS),
    '--save_iterations', str(DIAGNOSTIC_ITERATIONS),
    '--resolution', '64',
    '--eval'
]
print(' '.join(command))
try:
    completed = subprocess.run(command, cwd=GS_REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)
    output = completed.stdout or ''
except Exception as error:
    output = repr(error)
    completed = None
LOG_PATH.write_text(output, encoding='utf-8')
print(output[-20000:])
if completed is not None:
    print('Código de salida:', completed.returncode)
print('Log completo:', LOG_PATH)

=== Entrenamiento mínimo ===
/usr/bin/python3 train.py -s /content/truck -m /content/gs_diagnostic_vpl_5n_h --iterations 10 --test_iterations 10 --save_iterations 10 --resolution 64 --eval
2026-09-28 18:29:47.624894: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790620187.648681    7228 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790620187.654847    7228 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-28 18:29:47.675355: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in ot

In [6]:
print('=== Interpretación ===')
log = LOG_PATH.read_text(encoding='utf-8', errors='replace') if LOG_PATH.exists() else ''
patterns = [
    ('CUDA out of memory', 'Falta VRAM. Mantén --resolution 64/256, reduce vistas o usa una GPU con más VRAM.'),
    ('CUDA error', 'Hay un problema de compatibilidad CUDA/PyTorch/extensión compilada. Reinicia la sesión y recompila en el mismo runtime.'),
    ('No module named', 'Falta una dependencia o el kernel usa otro Python. Instala usando sys.executable.'),
    ('undefined symbol', 'La extensión fue compilada con una versión distinta de PyTorch/CUDA. No mezcles instalaciones; reinicia y reinstala.'),
    ('Could not load the custom C++ ops', 'torchvision o una extensión C++ es incompatible con PyTorch.'),
    ('FileNotFoundError', 'La ruta del dataset o de sparse/0 no es válida.'),
    ('does not exist', 'Falta una carpeta/archivo del dataset o la ruta usada no coincide con Drive.'),
    ('No CUDA GPUs are available', 'La sesión no tiene GPU activa.'),
]
if 'Training complete.' in log and 'Evaluating test' in log:
    print('DIAGNÓSTICO APROBADO: train.py terminó correctamente, evaluó test y guardó el modelo.')
    found = True
else:
    found = False
for pattern, advice in patterns:
    if pattern.lower() in log.lower():
        print('Coincidencia:', pattern)
        print('Siguiente acción:', advice)
        found = True
if not found and log:
    print('No reconocí automáticamente el error. Comparte el contenido de', LOG_PATH)
elif not log:
    print('No existe log: la celda de entrenamiento no alcanzó a ejecutarse.')

=== Interpretación ===
No reconocí automáticamente el error. Comparte el contenido de /content/gs_diagnostic_train.log
